# Streaming Responses in Agentic Platform

## Why Streaming Matters

LLM agents routinely take 5–15 seconds to generate a full response — and multi-step agents with tool calls can take 30+ seconds. Without streaming, the user stares at a blank screen the entire time. Streaming fixes this by pushing tokens and events the instant they're generated.

| | HTTP (wait-for-complete) | SSE (streaming) |
|--|--------------------------|-----------------|
| **Time-to-first-token** | 5–15s (full generation must finish) | 200–500ms |
| **User perception** | "Is this thing broken?" | "It's thinking and responding" |
| **Tool call visibility** | Nothing until done | "Searching web…" appears in real-time |
| **Timeout risk** | Long generations hit gateway/LB timeouts (30–60s) | Continuous data flow keeps connection alive |
| **Server memory** | Buffer entire response in RAM | Yield chunks, GC as you go |

**Bottom line:** Streaming turns a 10-second dead wait into an interactive typing experience. For multi-step agents, it's the difference between a loading spinner and a transparent workflow where users see each step happen.

## Why SSE (not WebSockets)?

Server-Sent Events are **unidirectional** (server→client) — which is exactly what LLM streaming needs. Compared to WebSockets:
- Works over plain HTTP/1.1 (no upgrade handshake)
- Compatible with standard load balancers, CDNs, and proxies
- Auto-reconnects on network drops (built into the browser `EventSource` API)
- Much simpler server implementation (just `yield` strings)

WebSockets add bidirectional complexity you don't need when the client only sends one request and then listens.

## Event Types

Our platform streams these events via SSE:
- `text_delta`: A chunk of text content
- `thinking_delta`: Agent's internal reasoning (chain-of-thought)
- `tool_call`: Tool being called
- `tool_result`: Result from tool
- `error`: Error information
- `done`: Stream completion

Each event includes a unique ID, conversation ID, timestamp, usage statistics (for LLM responses), and additional metadata.

## SSE Wire Format

```
data: {"type": "text_delta", "text": "Based on"}\n\n
data: {"type": "text_delta", "text": " my search"}\n\n
data: {"type": "tool_call", "tool_call": {...}}\n\n
data: {"type": "done"}\n\n
```

Each event is prefixed with `data: `, followed by a JSON payload, terminated by a double newline (`\n\n`). This is the SSE protocol spec — the double newline is what tells the browser "this event is complete, dispatch it."

To get started, lets take a look at our streaming types

## Available Event Classes

All streaming events inherit from `BaseStreamEvent` which provides: `id`, `session_id`, `timestamp`, and `metadata`.

| Class | Type Key | Purpose | Key Fields |
| --- | --- | --- | --- |
| `StartEvent` | `start` | Signals the stream has begun | — |
| `TextDeltaEvent` | `text_delta` | A chunk of generated text | `text: str` |
| `ThinkingDeltaEvent` | `thinking_delta` | Agent's chain-of-thought reasoning | `thinking: str` |
| `ContentBlockStart` | `content_block_start` | Marks the start of a content block | `content_type`, `content_block` |
| `ContentBlockEnd` | `content_block_end` | Marks the end of a content block | — |
| `ToolCallEvent` | `tool_call` | A complete tool invocation | `tool_call: ToolCall` |
| `ToolCallDeltaEvent` | `tool_call_delta` | Incremental tool call arguments | `arguments_delta: str` |
| `ToolResultEvent` | `tool_result` | Result returned from a tool | `tool_result: ToolResult` |
| `ErrorEvent` | `error` | An error occurred during streaming | `error: str` |
| `DoneEvent` | `done` | Signals the stream is complete | — |

These are defined in `agentic_platform.core.models.streaming_models` and discriminated by their `type` field for easy deserialization.

In [ ]:
from agentic_platform.core.models.streaming_models import ToolCallEvent, ToolResultEvent, ErrorEvent, DoneEvent, TextDeltaEvent, ThinkingDeltaEvent


TextDeltaEvent??

In [ ]:
TextDeltaEvent??

In [ ]:
ToolCallEvent??

Looking at the events above, you can see they all follow a similar flow. Text events have a text field, tool call and tool results have outputted results, etc.. Importantly, each event has a "type" attribute which makes it easier to piece together agent responses on a frontend.

Next lets build a simple agent.

In [ ]:
from labs_common import HAIKU_STRANDS_ID
# Next lets create our researcher agent. 
from pydantic_ai import Agent as PyAIAgent

import nest_asyncio
nest_asyncio.apply()


# Create a basic agent with a specialized system prompt
agent = PyAIAgent(
    HAIKU_STRANDS_ID,
    system_prompt="You are a helpful assistant."
)

# The response will be automatically printed by the Agent class
ABSTRACTION_QUESTION = "Explain the concept of abstractions in programming in one paragraph."
response = agent.run_sync(ABSTRACTION_QUESTION)
response.output

The agent above is a simple agent that doesn't do much. one thing you might have noticed in all the agents we've built up to this point is that they take a while to return. End users are often impatient and don't want to wait 10-15 seconds for a response. A way to make the latency less noticable is by streaming results back to the user. Most frameworks support streaming.

In the example below we'll stream the text results back from the agent output it as it comes in.

In [ ]:
async with agent.run_stream(ABSTRACTION_QUESTION) as result:
    async for message in result.stream_text(delta=True):  
            print(message)

Using streaming, the latency is much less noticable. However, what if we want to stream intermediate actions back? 

To do that, we'll need to return the structured output as well. Lets add a simple tool to our agent and stream the intermediate tool results back. 

In [ ]:
def get_weather(location: str) -> str:
    '''Useful for getting the local weather'''
    return f'The weather in {location} is Sunny and 70 degrees.'

In [ ]:
agent.tool_plain(get_weather)

In [ ]:
from pydantic_core import to_jsonable_python

nodes = []
async with agent.iter('What is the weather in SF?') as result:
    async for message in result:   
        nodes.append(to_jsonable_python(message))

for n in nodes:
    print(n)

Nice, we're now getting intermediate results out of our pydantic agent. However, we want to convert this into our streaming types to decouple the rest of our code from the specific framework. We've created a converter for this which we'll import below.

In [ ]:
from agentic_platform.core.converter.pydanticai_converters import PydanticAIStreamingEventConverter
from agentic_platform.core.models.streaming_models import StreamEvent
from typing import List

for node in nodes:
    events: List[StreamEvent] = PydanticAIStreamingEventConverter.convert_event(node, session_id='abc123')
    if events:
        for event in events:
            print(event)

Perfect. Now that we have our streaming events, it's time to bring it all together. Lets re-run our agent but this tiime lets convert into our message types and stream them back. As part of SSE, data is expected to be prefixed with "data: [YOUR DATA HERE]" folowed by two new lines. 

To stream results back, we'll use the yield. Yield allows allows for real-time streaming without buffering all the events.

In [ ]:
import nest_asyncio
import json
from fastapi import FastAPI
from pydantic import BaseModel
from pydantic_core import to_jsonable_python
from typing import AsyncGenerator
import uuid

nest_asyncio.apply()

# Import your converter
from agentic_platform.core.models.api_models import AgenticRequestStream

class AgentRequest(BaseModel):
    prompt: str
    conversation_id: str = "default"

async def generate_agent_events(request: AgenticRequestStream) -> AsyncGenerator[str, None]:
    """Generate Server-Sent Events from the agent stream"""

    session_id: str = request.session_id if request.session_id else str(uuid.uuid4())

    async with agent.iter(request.message.text) as result:
        async for message in result:   
            json_message = to_jsonable_python(message)
            events: List[StreamEvent] = PydanticAIStreamingEventConverter.convert_event(json_message, session_id)
            for event in events:
                sse_data = f"data: {json.dumps(event.model_dump_json(serialize_as_any=True))}\n\n"
                yield sse_data

In [ ]:
async for sse_data in generate_agent_events(AgenticRequestStream.from_text(text="What is the weather in SF?", **{'session_id':"abc123"})):
    print(sse_data)

## What We Built

In this notebook we went from a blocking LLM call to a production-ready SSE streaming pipeline:

1. **Blocking call** — `agent.run_sync()` returns only after the full response is generated (5–15s dead wait)
2. **Text streaming** — `agent.run_stream()` yields tokens as they're produced (200ms to first token)
3. **Structured event streaming** — `agent.iter()` exposes intermediate steps (tool calls, tool results) as they happen
4. **Framework-agnostic events** — `PydanticAIStreamingEventConverter` translates framework-specific nodes into our platform's `StreamEvent` types
5. **SSE wire format** — `generate_agent_events()` wraps everything in `data: {...}\n\n` format ready for FastAPI's `StreamingResponse`

## Key Patterns to Remember

| Pattern | When to Use |
|---------|-------------|
| `run_sync()` | Background jobs, batch processing — latency doesn't matter |
| `run_stream()` (text only) | Simple chat UIs that only need the final text |
| `iter()` + converter | Production agents where users need to see tool calls, reasoning, and progress |
| SSE (`yield` + `StreamingResponse`) | Any HTTP endpoint serving real-time agent output to a frontend |

## Production Considerations

- **Timeouts**: SSE keeps the connection alive with continuous data flow — but set a reasonable server-side timeout (e.g., 5 minutes) to prevent zombie connections.
- **Reconnection**: The browser `EventSource` API auto-reconnects. Include a `session_id` so the client can resume or replay missed events.
- **Error handling**: Always emit an `ErrorEvent` before closing the stream on failure — don't just drop the connection silently.
- **Backpressure**: If the client can't consume events fast enough, consider buffering or dropping `thinking_delta` events (they're informational, not critical).
- **Observability**: Log `session_id` + event counts per stream. A stream that emits zero `text_delta` events likely hit an error path.

